### Imports and data preparation

In [1]:
from core import utils, database, paths
import pandas as pd
import datetime
import requests


In [ ]:

data : pd.DataFrame | None = database.DatabaseManager().run_query("""select * from tesco_analysts.pmajor1_wtd_for_daily_tableau_test""")
data.to_csv(paths.ASSETS_PATH + f"\daily_test_data.csv", index=False)

today = datetime.date.today().strftime('%Y-%m-%d')



In [5]:
def analyse_financial_summaries(system_prompt, user_data, model='martain7r/finance-llama-8b:q4_k_m'):
    url = "http://localhost:11434/api/chat" # Fontos: a /api/chat végpontot használjuk!
    
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_data}
        ],
        "stream": False,
        "options": {
            "format": "json",
            "temperature": 0.15,
            "top_p": 0.9,
            "num_predict": 800,     # JSON-hoz általában elég
            "num_ctx": 4096,
            "seed": 42,
            "repeat_penalty": 1.18,
            "repeat_last_n": 256,
            "presence_penalty": 0.1,
            "frequency_penalty": 0.1,
            "stop": ["}\n"]         # ha a modell hajlamos kódfence-re
        }
}
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        # A válasz szerkezete itt kicsit más: message['content']
        return response.json().get("message", {}).get("content", "Nothing to analyze")
    else:
        return f"Error: {response.status_code} - {response.text}"

In [2]:
today

'2026-01-28'

## Version 1

In [ ]:
data = pd.read_csv(paths.ASSETS_PATH + f"\daily_test_data.csv")
mod_data = data.copy()
day = mod_data['day'].unique().max()

# If 'day' is a single value (e.g., 20250531), convert to date:
day_int = int(day)  # e.g., 20250531
year = day_int // 10000
month = (day_int % 10000) // 100
d = day_int % 100


month_name = datetime.date(year, month, d).strftime("%m.%d")
last_day_mod_data = mod_data[mod_data['day'] == mod_data['day'].max()]
actual_data = last_day_mod_data

len(mod_data['day'].unique())
longer_period_data = mod_data
def data_collector(actual_data : pd.DataFrame):
    
    if len(actual_data['day']) >1:
        intervall = 'last day'
    else:
        intervall = 'last 21 days'

    aggregator = {
        'sales_ty': ['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo',
                'sales_ex_vat_ty_clearance', 'sales_ex_vat_ty_others'],
        'sales_ly': ['sales_ex_vat_ly_standard', 'sales_ex_vat_ly_promo',
                    'sales_ex_vat_ly_clearance', 'sales_ex_vat_ly_others'],
        'sales_lfl' : [ 'sales_lfl_standard', 'sales_lfl_promo', 'sales_lfl_clearance','sales_lfl_others'],
        'sales_lflb' : ['sales_lflb_standard', 'sales_lflb_promo', 'sales_lflb_clearance','sales_lflb_others'],
        'margin_ty' : ['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance','margin_ty_others'],
        'unit_ty' : [ 'units_ty_standard', 'units_ty_promo','units_ty_clearance', 'units_ty_others'],
        'sales_ly' : ['sales_ex_vat_ly_standard','sales_ex_vat_ly_promo', 'sales_ex_vat_ly_clearance','sales_ex_vat_ly_others'],
        'margin_ly' : ['margin_ly_standard', 'margin_ly_promo','margin_ly_clearance', 'margin_ly_others'],
        'unit_ly' : [ 'units_ly_standard','units_ly_promo', 'units_ly_clearance', 'units_ly_others'],
        'margin_lfl' : ['margin_lfl_standard', 'margin_lfl_promo','margin_lfl_clearance', 'margin_lfl_others'],
        'unit_lfl' : ['units_lfl_standard','units_lfl_promo', 'units_lfl_clearance', 'units_lfl_others'],
        'margin_lflb' : ['margin_lflb_standard', 'margin_lflb_promo','margin_lflb_clearance', 'margin_lflb_others'],
        'unit_lflb' : [ 'units_lflb_standard','units_lflb_promo', 'units_lflb_clearance', 'units_lflb_others'],
        'sales_2ylfl' : ['sales_2ylfl_standard', 'sales_2ylfl_promo', 'sales_2ylfl_clearance','sales_2ylfl_others'],
        'margin_2ylfl' : ['margin_2ylfl_standard', 'margin_2ylfl_promo','margin_2ylfl_clearance', 'margin_2ylfl_others'],
        'unit_2ylfl' : ['units_2ylfl_standard','units_2ylfl_promo', 'units_2ylfl_clearance', 'units_2ylfl_others'],
        'sales_2ylflb' : [ 'sales_2ylflb_standard', 'sales_2ylflb_promo', 'sales_2ylflb_clearance','sales_2ylflb_others'],
        'margin_2ylflb' : ['margin_2ylflb_standard', 'margin_2ylflb_promo','margin_2ylflb_clearance', 'margin_2ylflb_others'],
        'unit_2ylflb' : ['units_2ylflb_standard', 'units_2ylflb_promo', 'units_2ylflb_clearance','units_2ylflb_others'],'sales_budget' : ['sales_budget'],
        'cols' : ['country', 'department', 'department_name','section', 'section_name', 'group', 'group_name', 'subgroup','subgroup_name']
        
    }

    cols = aggregator['cols']

    budget_df = mod_data.drop_duplicates(['sales_budget'])
    actual_budget = actual_data.drop_duplicates(['sales_budget'])

    def grouper_totaler(df : pd.DataFrame,group_index=len(cols) ,only_total: bool=True, values : list = aggregator['sales_ty']):
        #--- Create grouping by Country ---
        needed_cols = cols[:group_index]
        df2 = df.groupby(by=needed_cols)[values].sum().reset_index().sort_values([needed_cols[0],], ascending=False)
        prefix = values[0].split("_")[:2]
        df2[f'Total_{prefix}'] = df2[values].sum(axis=1)
        df2.columns = df2.columns.str.replace(r"[\[\]\.,'\"]", "", regex=True)
        df2.columns = df2.columns.str.replace("[' ']","_", regex=True)
        if only_total:
            return df2.loc[:, needed_cols + [df2.columns[-1]]]
        else:
            return df2

    total_sales_lfl = grouper_totaler(actual_data,group_index=1,values = aggregator['sales_lfl'])
    total_sales_lflb = grouper_totaler(actual_data, group_index=1, values=aggregator['sales_lflb'])
    total_sales_by_country = grouper_totaler(actual_data, group_index=1, values=aggregator['sales_ty'])

    #TOTAL budget and forecast numbers
    sales_forecast = actual_budget['daily_sales_forecast'].sum()
    sales_budget = actual_budget['sales_budget'].sum()
    total_sales = total_sales_by_country.iloc[:,1].sum()

    #Versus numbers
    vs_budget = total_sales - sales_budget
    vs_budget_perc = (total_sales / sales_budget -1)*100
    vs_forecast = total_sales - sales_forecast
    vs_forecast_perc = (total_sales / sales_forecast -1) *100 


    best_sales_country = total_sales_by_country.sort_values(by=total_sales_by_country.columns[1],ascending=False).iloc[0,0] 



    total_promo_sales_lfl = actual_data.groupby('country')['sales_lfl_promo'].sum().reset_index()
    total_promo_sales_lflb = actual_data.groupby('country')['sales_lflb_promo'].sum().reset_index()

    total_clearance_sales_lfl = actual_data.groupby('country')['sales_lfl_clearance'].sum().reset_index()
    total_clerance_sales_lflb = actual_data.groupby('country')['sales_lflb_clearance'].sum().reset_index()


    def lflzator(df1 : pd.DataFrame,df2 : pd.DataFrame):
        df3 = df1.merge(df2, on='country')
        df3['Sales_lfl'] = df1.iloc[:,1] - df2.iloc[:,1]
        df3['Total_sales_lfl%'] = (df1.iloc[:,1] / df2.iloc[:,1] -1 )*100
        df3 = df3.sort_values('Sales_lfl', ascending=False)
        return df3.iloc[:, [0,3,4]]

    def extract_top_lfl(df: pd.DataFrame, top_n: int = 3) -> dict:
        """
        Visszaadja a legjobb országokat és az LFL értékeiket.

        Args:
            df (pd.DataFrame): Kétszlopos dataframe, ahol az első az ország, a második az LFL érték.
            top_n (int): Hány országot adjon vissza (alapértelmezés 3).

        Returns:
            dict: Például {'best': ('CZ', 3.2), 'second': ('HU', 1.8), 'third': ('SK', 0.9)}
        """
        result = {}
        ranks = ['best', 'second', 'third', 'fourth', 'fifth']
        for i in range(min(top_n, len(df))):
            result[ranks[i]] = (df.iloc[i, 0], df.iloc[i, 2])
        return result

    total_sales_lfl_real = lflzator(total_sales_lfl, total_sales_lflb)
    total_promo_sales_lfl_real = lflzator(total_promo_sales_lfl,total_promo_sales_lflb)
    total_clearance_sales_lfl_real = lflzator(total_clearance_sales_lfl, total_clerance_sales_lflb)

    country_lfl = extract_top_lfl(total_sales_lfl_real,3)
    best_lfl_country, best_lfl_country_value = country_lfl['best']
    second_lfl_country, second_lfl_country_value = country_lfl['second']
    third_lfl_country, third_lfl_country_value = country_lfl['third']

    country_promo_lfl = extract_top_lfl(total_promo_sales_lfl_real)
    best_promo_lfl_country, best_promo_lfl_country_value = country_promo_lfl['best']
    second_promo_lfl_country, second_promo_lfl_country_value = country_promo_lfl['second'] 
    third_promo_lfl_country, third_promo_lfl_country_value = country_promo_lfl['third']

    country_clearance_lfl = extract_top_lfl(total_promo_sales_lfl_real)
    best_clearance_lfl_country, best_clearance_lfl_country_value = country_clearance_lfl['best']
    second_clearance_lfl_country, second_clearance_lfl_country_value = country_clearance_lfl['second'] 
    third_clearance_lfl_country, third_clearance_lfl_country_value = country_clearance_lfl['third']

    total_ce_lfl = (total_sales_lfl['Total_sales_lfl'].sum() / total_sales_lflb['Total_sales_lflb'].sum() -1 )*100
    ce_promo_lfl = (actual_data['sales_lfl_promo'].sum() / actual_data['sales_lflb_promo'].sum()-1)*100 
    ce_range_lfl = (actual_data['sales_lfl_standard'].sum() / actual_data['sales_lflb_standard'].sum()-1)*100 
    ce_clearance_lfl = (actual_data['sales_lfl_clearance'].sum() / actual_data['sales_lflb_clearance'].sum()-1)*100 


    total_sales_string = f"""
    You are a financial analyst comparing TODAY's performance against the 21-DAY TREND for the Central Europe GM division.

    STRICT INSTRUCTIONS:
    1. Compare each daily figure to its 21-day trend equivalent.
    2. Identify if today is an acceleration or a slowdown compared to the last 3 weeks.
    3. Use 4-5 concise sentences. Each sentence must start in a new line.

    DATA FOR ANALYSIS (Today vs 21-Day Trend):
    - Daily Sales: {total_sales:,.0f}£ (Trend average: {trend_total_sales:,.0f}£)
    - Daily Budget Var: {vs_budget_perc:.2f}% (Trend average: {trend_vs_budget_perc:.2f}%)
    - Daily Total LFL: {total_ce_lfl:.2f}% (Trend average: {trend_ce_lfl:.2f}%)
    - Daily Promo LFL: {ce_promo_lfl:.2f}% (Trend average: {trend_promo_lfl:.2f}%)
    - Top Country Today: {best_lfl_country} ({best_lfl_country_value:.2f}% LFL)
    - Top Country Trend: {trend_best_lfl_country} ({trend_best_lfl_value:.2f}% LFL)

    ANALYSIS QUESTIONS:
    - How does today's sales and budget variance compare to the 21-day momentum?
    - Is the LFL growth (especially Promo/Range) strengthening or weakening today?
    - Which country is over-performing its own 3-week trend?

    OUTPUT FORMAT:
    Professional financial narrative. No bullet points. Line breaks between insights.
    """
    return utils.analyse_financial_summaries(prompt_text=total_sales_string,model='gemma:2b')
actual_data_result = data_collector(actual_data)
longer_period_result = data_collector(longer_period_data)
actual_data_result
longer_period_result
summarizes =f"""You are a financial analyst creating a short executive summary for senior management about Central Europe GM division (CE: HU, SK, CZ)
               here you can read the last day's summary:\n {actual_data_result} and here is the last 21 days performance:\n {longer_period_result}.
               Focusing and highlighting the last day's summary for insight and use the last 21 days performance for trend""" 
summarizes
total_summary = utils.analyse_financial_summaries(prompt_text=summarizes,model='gemma:2b')
total_summary
import os
def write_out(szoveg):

    output_file = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_total_summary.md")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# AI Daily Summary\n\n")
        f.write(total_summary)
        f.write("\n")

    print(f"Markdown summary successfully saved to: {output_file}")
write_out(total_summary)

## Version 2

In [46]:
from core import utils, database, paths
import pandas as pd
import datetime
import os


In [ ]:

# 1. ADATGYŰJTÉS ÉS ELŐKÉSZÍTÉS
db_manager = database.DatabaseManager()
query = "select * from tesco_analysts.pmajor1_wtd_for_daily_tableau_test"
data = db_manager.run_query(query)


In [ ]:
data = pd.read_csv(paths.ASSETS_PATH + f"\daily_test_data.csv")
# Dátumkezelés: Biztosítjuk, hogy datetime formátumban legyen
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

# Szelektálás: Mai nap és az elmúlt 21 nap (trend) adatai
actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]

def calculate_detailed_metrics(df: pd.DataFrame):
    """Kiszámolja a pénzügyi mutatókat (Sales, Margin, LFL)."""
    
    # Sales aggregáció (TY = This Year)
    sales_ty = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 
                   'sales_ex_vat_ty_clearance', 'sales_ex_vat_ty_others']].sum().sum()
    
    # Margin aggregáció (TY = This Year)
    margin_ty = df[['margin_ty_standard', 'margin_ty_promo', 
                    'margin_ty_clearance', 'margin_ty_others']].sum().sum()
    
    # Költségvetés és Előrejelzés (egyedi értékek összege)
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()
    
    # Margin Rate (Árrés százalék)
    margin_rate = (margin_ty / sales_ty * 100) if sales_ty != 0 else 0
    
    # LFL Függvény (TY / LY bázis)
    def get_lfl(prefix):
        ty = df[[f'{prefix}_lfl_standard', f'{prefix}_lfl_promo', f'{prefix}_lfl_clearance']].sum().sum()
        ly = df[[f'{prefix}_lflb_standard', f'{prefix}_lflb_promo', f'{prefix}_lflb_clearance']].sum().sum()
        return ((ty / ly) - 1) * 100 if ly != 0 else 0

    # Speciális LFL-ek
    clearance_lfl = (df['sales_lfl_clearance'].sum() / df['sales_lflb_clearance'].sum() - 1) * 100 if df['sales_lflb_clearance'].sum() != 0 else 0
    promo_lfl = (df['sales_lfl_promo'].sum() / df['sales_lflb_promo'].sum() - 1) * 100 if df['sales_lflb_promo'].sum() != 0 else 0

    # Országos bontás (Top 3 LFL alapján)
    country_lfl = df.groupby('country').apply(
        lambda x: (x['sales_lfl_standard'].sum() / x['sales_lflb_standard'].sum() - 1) * 100 if x['sales_lflb_standard'].sum() != 0 else 0
    ).sort_values(ascending=False).head(3).to_dict()

    return {
        'sales': sales_ty,
        'margin_rate': margin_rate,
        'margin_lfl': get_lfl('margin'),
        'sales_lfl': get_lfl('sales'),
        'clearance_lfl': clearance_lfl,
        'promo_lfl' : promo_lfl,
        'vs_budget_perc': ((sales_ty / budget - 1) * 100) if budget != 0 else 0,
        'vs_forecast_perc': ((sales_ty / forecast - 1) * 100) if forecast != 0 else 0,
        'top_countries': country_lfl
    }

# 2. METRIKÁK KISZÁMÍTÁSA
actual_metrics = calculate_detailed_metrics(actual_df)
trend_metrics = calculate_detailed_metrics(trend_df)

system_message = (
    "You are a Group Financial Controller writing for an Executive Committee. "
    "Your primary objectives are factual accuracy and numeric consistency. "
    "Avoid narrative, storytelling, and speculation. "
    "Never introduce any numbers beyond those explicitly provided in the user content. "
    "If a claim cannot be supported numerically from the provided data, omit it. "
    "Prefer concise, analytical sentences (max 25 words). "
    "Any speculative or unsupported statement is a critical error."
)

# 3. FINANCE-LLAMA PROMPT ÖSSZEÁLLÍTÁSA
# A modell neve: martain7r/finance-llama-8b:q4_k_m

prompt = f"""
STRICT RULE: Every insight must include the specific numbers and the delta (change) between Today and the Trend.

FINANCIAL DATA (Today vs 21-Day Trend):
- Net Sales: {actual_metrics['sales']:,.0f}£ (Trend Avg: {trend_metrics['sales']/21:,.0f}£) 
- Budget Variance: {actual_metrics['vs_budget_perc']:.2f}% (Trend: {trend_metrics['vs_budget_perc']:.2f}%)
- Margin Rate: {actual_metrics['margin_rate']:.2f}% (Trend: {trend_metrics['margin_rate']:.2f}%)
- Margin LFL: {actual_metrics['margin_lfl']:.2f}% (Trend: {trend_metrics['margin_lfl']:.2f}%)
- Sales LFL: {actual_metrics['sales_lfl']:.2f}% (Trend: {trend_metrics['sales_lfl']:.2f}%)
- Promo Sales LFL: {actual_metrics['promo_lfl']:.2f}% (Trend: {trend_metrics['promo_lfl']:.2f}%)
- Clearance LFL: {actual_metrics['clearance_lfl']:.2f}% (Trend: {trend_metrics['clearance_lfl']:.2f}%)
- Top Country: HU {actual_metrics['top_countries'].get('HU', 0):.1f}% LFL

DETAILED ANALYSIS STEPS:
1. Compare Today's Budget Var ({actual_metrics['vs_budget_perc']:.2f}%) to the Trend ({trend_metrics['vs_budget_perc']:.2f}%). Is it an overperformance?
2. Analyze the Margin Rate uplift. Today is {actual_metrics['margin_rate']:.2f}% vs Trend {trend_metrics['margin_rate']:.2f}%. Calculate the basis point (bps) improvement.
3. Address the "Clearance Trap": Today's Clearance LFL is {actual_metrics['clearance_lfl']:.2f}%, which is significantly higher than the {trend_metrics['clearance_lfl']:.2f}% trend. How did this affect the Sales LFL ({actual_metrics['sales_lfl']:.2f}%)?
4. Performance Gap: Note that Margin LFL ({actual_metrics['margin_lfl']:.2f}%) is lagging behind the 21-day trend ({trend_metrics['margin_lfl']:.2f}%). Explain what this means for bottom-line health.

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}


OUTPUT FORMAT:
- Title: # Data-Driven Strategic Summary
- Professional narrative that INTEGRATES the numbers into the sentences.
- Use bold text for the most critical deltas.
"""


# 4. LLM HÍVÁS ÉS MENTÉS
final_summary = utils.analyse_financial_summaries(user_data=prompt,system_prompt=system_message, model='martain7r/finance-llama-8b:q4_k_m')

def save_report(content):
    output_path = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_report_VERSION2_1.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Report saved to: {output_path}")

save_report(final_summary)

C:\Users\pmajor1\AppData\Local\Temp\ipykernel_9352\963359615.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  country_lfl = df.groupby('country').apply(
C:\Users\pmajor1\AppData\Local\Temp\ipykernel_9352\963359615.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  country_lfl = df.groupby('country').apply(


Report saved to: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_report_VERSION2_1.md


## Version 3

In [ ]:
from core import utils, database, paths
import pandas as pd
import datetime
import os

# 1. ADATELŐKÉSZÍTÉS
db_manager = database.DatabaseManager()
query = "SELECT * FROM tesco_analysts.pmajor1_wtd_for_daily_tableau_test"
data = db_manager.run_query(query)


In [66]:
# 1. ADATELŐKÉSZÍTÉS
data = pd.read_csv(paths.ASSETS_PATH + r"\daily_test_data.csv")
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]

def get_complete_metrics(df: pd.DataFrame):
    """Kiszámolja az összesített és országos szintű mutatókat, irányjelzőkkel."""
    
    # Segédfüggvény az LFL és irány meghatározásához
    def calc_lfl_info(ty_col, ly_col):
        ty_sum = df[ty_col].sum()
        ly_sum = df[ly_col].sum()
        if ly_sum == 0: return "0.0% flat"
        val = (ty_sum / ly_sum - 1) * 100
        direction = "increase" if val >= 0 else "decrease"
        return f"{val:.1f}% {direction}"

    # --- TOTAL CE SZINT ---
    total_sales = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
    total_margin = df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum()
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()
    
    metrics = {
        'sales': total_sales,
        'margin_rate': (total_margin / total_sales * 100) if total_sales != 0 else 0,
        'vs_budget_perc': ((total_sales / budget - 1) * 100) if budget != 0 else 0,
        'vs_forecast_perc' : ((total_sales / forecast - 1) * 100) if forecast != 0 else 0,
        'range_share' : (df['sales_ex_vat_ty_standard'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'promo_share': (df['sales_ex_vat_ty_promo'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'clearance_share': (df['sales_ex_vat_ty_clearance'].sum() / total_sales * 100) if total_sales != 0 else 0,
    }

    # --- ORSZÁGOS SZINT ---
    countries = ['HU', 'CZ', 'SK']
    metrics['countries'] = {}
    
    for c in countries:
        c_df = df[df['country'] == c]
        if not c_df.empty:
            # Itt már a segédfüggvényt használjuk, hogy az LLM ne tévesszen előjelet
            c_sales = c_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['countries'][c] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard'),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo'),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance'),
                'margin_rate': (c_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if c_sales != 0 else 0
            }
    return metrics

actual_m = get_complete_metrics(actual_df)
trend_m = get_complete_metrics(trend_df)

# 2. STRUKTURÁLT PROMPT (KIEGÉSZÍTETT INSTRUKCIÓKKAL)
prompt = f"""
You are a Senior Financial Analyst. Analyze the Central Europe (CE) GM Division performance.
Compare TODAY vs the 21-DAY TREND. 

### SECTION 1: CE OVERVIEW
- Net Sales: Today {actual_m['sales']:,.0f}£ (Trend Avg: {trend_m['sales']/21:,.0f}£)
- Budget Var: Today {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Var: Today {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: Today {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%)
- Sales Mix (Range/Promo/Clearance Share): Today {actual_m['range_share']:.1f}%/{actual_m['promo_share']:.1f}%/{actual_m['clearance_share']:.1f}% vs Trend {trend_m['range_share']:.1f}%/{trend_m['promo_share']:.1f}%/{trend_m['clearance_share']:.1f}%

### SECTION 2: COUNTRY PERFORMANCE (HU, CZ, SK)
- HU: Range LFL: {actual_m['countries']['HU']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range LFL: {actual_m['countries']['CZ']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range LFL: {actual_m['countries']['SK']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['SK']['clearance_lfl_txt']}

### INSTRUCTIONS:
1. Identify the main CE driver (HU, CZ or SK).
2. Pay close attention to the "increase/decrease" tags provided in the data. Never interpret an "increase" as a negative performance.
3. Analyze the 'Sales Mix shift': Is the margin rate improving or eroding due to the Promo/Clearance mix?
4. For each country, provide a 1-sentence insight explaining if their performance is healthy (range-driven) or risky (clearance-driven).

OUTPUT FORMAT:
- Use clear headings: # CE Overview, # Key Regional Drivers, # Country Insights.
- Professional financial narrative. Bold the most important percentages.
"""

# 3. LLM HÍVÁS ÉS MENTÉS
total_summary = utils.analyse_financial_summaries(prompt_text=prompt, model='martain7r/finance-llama-8b:q4_k_m')

def write_out(content):
    output_file = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_complete_v2.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content)
    print(f"Report successfully saved: {output_file}")

write_out(total_summary)

Report successfully saved: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_complete_v2.md


In [ ]:
from core import utils, database, paths
import pandas as pd
import datetime
import os

# 1. ADATELŐKÉSZÍTÉS
data = pd.read_csv(paths.ASSETS_PATH + r"\daily_test_data.csv")
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]

def get_complete_metrics(df: pd.DataFrame):
    """Kiszámolja az összesített, országos és osztály szintű mutatókat."""
    
    def calc_lfl_info(ty_col, ly_col, input_df=None):
        target_df = input_df if input_df is not None else df
        ty_sum = target_df[ty_col].sum()
        ly_sum = target_df[ly_col].sum()
        if ly_sum == 0: return "0.0% flat"
        val = (ty_sum / ly_sum - 1) * 100
        direction = "increase" if val >= 0 else "decrease"
        return f"{val:.1f}% {direction}"

    # --- TOTAL CE SZINT ---
    total_sales = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
    total_margin = df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum()
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()
    
    metrics = {
        'sales': total_sales,
        'margin_rate': (total_margin / total_sales * 100) if total_sales != 0 else 0,
        'vs_budget_perc': ((total_sales / budget - 1) * 100) if budget != 0 else 0,
        'vs_forecast_perc' : ((total_sales / forecast - 1) * 100) if forecast != 0 else 0,
        'range_share' : (df['sales_ex_vat_ty_standard'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'promo_share': (df['sales_ex_vat_ty_promo'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'clearance_share': (df['sales_ex_vat_ty_clearance'].sum() / total_sales * 100) if total_sales != 0 else 0,
    }

    # --- ORSZÁGOS SZINT ---
    metrics['countries'] = {}
    for c in ['HU', 'CZ', 'SK']:
        c_df = df[df['country'] == c]
        if not c_df.empty:
            c_sales = c_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['countries'][c] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard', c_df),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo', c_df),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance', c_df),
                'margin_rate': (c_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if c_sales != 0 else 0
            }

    # --- DEPARTMENT SZINT (Top/Bottom 3) ---
    dept_gb = df.groupby('department_name').apply(
        lambda x: (x['sales_lfl_standard'].sum() / x['sales_lflb_standard'].sum() - 1) * 100 if x['sales_lflb_standard'].sum() != 0 else 0
    ).sort_values(ascending=False)
    
    metrics['top_depts'] = dept_gb.head(3).to_dict()
    metrics['bottom_depts'] = dept_gb.tail(3).to_dict()

    return metrics

actual_m = get_complete_metrics(actual_df)
trend_m = get_complete_metrics(trend_df)



system_message = (
    "You are a Group Financial Controller writing for an Executive Committee. "
    "Your primary objectives are factual accuracy and numeric consistency. "
    "Avoid narrative, storytelling, and speculation. "
    "Never introduce any numbers beyond those explicitly provided in the user content. "
    "If a claim cannot be supported numerically from the provided data, omit it. "
    "Prefer concise, analytical sentences (max 25 words). "
    "Any speculative or unsupported statement is a critical error."
)


# Csak azokat az adatokat adjuk át, amik kellenek a szövegbe, 
# és megmutatjuk a modellnek a várt stílust.
top_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['top_depts'].items()])
bottom_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['bottom_depts'].items()])


# Számítsunk ki 2-3 kulcsfontosságú összefüggést Pythonban, hogy ne az LLM-nek kelljen
margin_diff = actual_m['margin_rate'] - trend_m['margin_rate']
mix_improvement = trend_m['clearance_share'] - actual_m['clearance_share']

margin_bps = int((actual_m['margin_rate'] - trend_m['margin_rate']) * 100)

# KIZÁRÓLAG a tények, nulla magyarázat a modellnek, hogy mit csináljon a tagekkel
margin_improvement = actual_m['margin_rate'] - trend_m['margin_rate']
clearance_drop = trend_m['clearance_share'] - actual_m['clearance_share']
sales_gap = actual_m['sales'] - (trend_m['sales']/21)


# Itt visszatesszük a teljes kontextust



user_message = f"""
### PERFORMANCE DATA (Facts – do not create new numbers):
- Net Sales: {actual_m['sales']:,.0f}£ (21-Day Trend avg: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Variance: {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%) → Improvement vs trend: {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp ({margin_bps} bps)
- Sales Mix Today (Range/Promo/Clearance): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Sales Mix Trend (Range/Promo/Clearance): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%
- Clearance share change (Trend → Today): {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp
- Sales gap vs trend average: {actual_m['sales'] - (trend_m['sales']/21):,.0f}£

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### TASK (Executive summary with strict constraints):
Write an executive summary as JSON only.

### OUTPUT (JSON, STRICT):
Return ONLY valid JSON with this exact schema:
{{
  "p1": "<string>",
  "p2": "<string>",
  "p3": "<string>",
  "p4": "<string>"
}}
Constraints (mandatory):
- Each paragraph value MUST be a plain string with EXACTLY 3 sentences.
- Each sentence MUST be 25 words or fewer.
- EVERY sentence MUST include at least one numeric value WITH the correct unit (% / pp / £).
- "p1" MUST start with EXACTLY: "Central Europe's financial performance today".
- Do NOT add any keys or metadata. No markdown, no code fences, no comments, no trailing commas.
- Do NOT copy or restate any line from the Facts section; integrate numbers into sentences only.
- Keep units correct: use "pp" for share changes and "%" for rates; use "£" only for sales amounts.
- Never convert percentages or pp into currency; never convert currency into percentages or pp.
- Do NOT introduce any number that is not present above.
- Do NOT speculate on causes (no macro factors, no marketing execution, no consumer behavior).
- Do NOT use phrases like "could be", "may be", "might", "suggests", or "appears".
- Refer ONLY to "Today" and "21-Day Trend"; no other time horizons.
- Use professional terminology: "margin dilution", "inventory velocity", "baseline demand".

Paragraph-to-key mapping (mandatory logic anchors):
- p1 (CE vs Budget/Forecast and margin improvement):
  • Compare Today Net Sales {actual_m['sales']:,.0f}£ to Trend avg {trend_m['sales']/21:,.0f}£ and state Budget {actual_m['vs_budget_perc']:.2f}% and Forecast {actual_m['vs_forecast_perc']:.2f}% variances.
  • Explain the margin improvement of {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp by referencing the mix shift: Clearance {actual_m['clearance_share']:.1f}% vs {trend_m['clearance_share']:.1f}% (reduced margin dilution).
  • All three sentences must be numeric and within 25 words.

- p2 (HU success vs CZ/SK using ONLY LFL texts and margin rates):
  • Use HU/CZ/SK LFL texts as given and country margin rates (HU {actual_m['countries']['HU']['margin_rate']:.2f}%, CZ {actual_m['countries']['CZ']['margin_rate']:.2f}%, SK {actual_m['countries']['SK']['margin_rate']:.2f}%).
  • Tie stronger HU LFLs to its margin rate numerically; contrast with CZ/SK strictly by numbers.
  • No additional drivers or interpretations.

- p3 (Sales Mix impact on profitability):
  • Quote Today vs Trend mix (Range {actual_m['range_share']:.1f}%/{trend_m['range_share']:.1f}%, Promo {actual_m['promo_share']:.1f}%/{trend_m['promo_share']:.1f}%, Clearance {actual_m['clearance_share']:.1f}%/{trend_m['clearance_share']:.1f}%).
  • State that lower Clearance by {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp reduced margin dilution and supported the {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp margin uplift.
  • Maintain numeric anchoring in all sentences.

- p4 (Departments: risks and opportunities):
  • Use EXACTLY the provided Top/Bottom lists with their % values; restate them compactly with numbers.
  • If a department name explicitly contains 'WIGIG' or 'Seasonal', you may label it as opportunity/risk respectively; otherwise DO NOT assign such labels.
  • Quantify risk/opportunity ONLY by the provided % values; no external justification.
"""





# A függvényed hívása (ügyelj rá, hogy a payload-ban a 'messages' listát használd)
total_summary = utils.analyse_financial_summaries(
    system_prompt=system_message, 
    user_data=user_message,
    model='martain7r/finance-llama-8b:q4_k_m'
)


def write_out(content):
    output_file = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_final_VERSION3_v22.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről
    print(f"Report saved: {output_file}")

write_out(total_summary)

C:\Users\pmajor1\AppData\Local\Temp\ipykernel_9352\947101178.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(
C:\Users\pmajor1\AppData\Local\Temp\ipykernel_9352\947101178.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(


Report saved: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_final_v22.md


In [5]:

from core import utils, database, paths
import pandas as pd
import datetime
import os
import requests
import json
import re
import time

# =========================================
# KONFIG (állítható)
# =========================================
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
MODEL_NAME = 'martain7r/finance-llama-8b:q4_k_m'

CONNECT_TIMEOUT = float(os.getenv("OLLAMA_CONNECT_TIMEOUT", "10"))   # mp
READ_TIMEOUT    = float(os.getenv("OLLAMA_READ_TIMEOUT", "600"))     # mp – első hívásnál legyen bőven
NUM_PREDICT     = int(os.getenv("OLLAMA_NUM_PREDICT", "420"))        # 4×3 mondathoz elég; ha lassú a gép: 360
NUM_CTX         = int(os.getenv("OLLAMA_NUM_CTX", "4096"))
SEED            = int(os.getenv("OLLAMA_SEED", "42"))

# =========================================
# 1) ADATELŐKÉSZÍTÉS
# =========================================
data = pd.read_csv(os.path.join(paths.ASSETS_PATH, "daily_test_data.csv"))
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]  # javítva (&gt; -> >)

# =========================================
# 2) METRIKA-GYŰJTŐ
# =========================================
def get_complete_metrics(df: pd.DataFrame):
    """Kiszámolja az összesített, országos és osztály szintű mutatókat."""
    def calc_lfl_info(ty_col, ly_col, input_df=None):
        target_df = input_df if input_df is not None else df
        ty_sum = target_df[ty_col].sum()
        ly_sum = target_df[ly_col].sum()
        if ly_sum == 0: return "0.0% flat"
        val = (ty_sum / ly_sum - 1) * 100
        direction = "increase" if val >= 0 else "decrease"
        return f"{val:.1f}% {direction}"

    # --- TOTAL CE SZINT ---
    total_sales = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
    total_margin = df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum()
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()

    metrics = {
        'sales': total_sales,
        'margin_rate': (total_margin / total_sales * 100) if total_sales != 0 else 0,
        'vs_budget_perc': ((total_sales / budget - 1) * 100) if budget != 0 else 0,
        'vs_forecast_perc': ((total_sales / forecast - 1) * 100) if forecast != 0 else 0,
        'range_share': (df['sales_ex_vat_ty_standard'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'promo_share': (df['sales_ex_vat_ty_promo'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'clearance_share': (df['sales_ex_vat_ty_clearance'].sum() / total_sales * 100) if total_sales != 0 else 0,
    }

    # --- ORSZÁGOS SZINT ---
    metrics['countries'] = {}
    for c in ['HU', 'CZ', 'SK']:
        c_df = df[df['country'] == c]
        if not c_df.empty:
            c_sales = c_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['countries'][c] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard', c_df),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo', c_df),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance', c_df),
                'margin_rate': (c_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if c_sales != 0 else 0
            }

    # --- DEPARTMENT SZINT (Top/Bottom 3) ---
    dept_gb = df.groupby('department_name').apply(
        lambda x: (x['sales_lfl_standard'].sum() / x['sales_lflb_standard'].sum() - 1) * 100 if x['sales_lflb_standard'].sum() != 0 else 0
    ).sort_values(ascending=False)

    metrics['top_depts'] = dept_gb.head(3).to_dict()
    metrics['bottom_depts'] = dept_gb.tail(3).to_dict()
    return metrics

# =========================================
# 3) METRIKÁK ELŐKÉSZÍTÉSE
# =========================================
actual_m = get_complete_metrics(actual_df)
trend_m  = get_complete_metrics(trend_df)

top_depts_str    = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['top_depts'].items()])
bottom_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['bottom_depts'].items()])

margin_bps          = int((actual_m['margin_rate'] - trend_m['margin_rate']) * 100)
margin_improvement  = actual_m['margin_rate'] - trend_m['margin_rate']
clearance_drop      = trend_m['clearance_share'] - actual_m['clearance_share']
sales_gap           = actual_m['sales'] - (trend_m['sales'] / 21)

# =========================================
# 4) PROMPTOK (JSON-mód)
# =========================================
system_message = (
    "You are a Group Financial Controller writing for an Executive Committee. "
    "Your primary objectives are factual accuracy and numeric consistency. "
    "Avoid narrative, storytelling, and speculation. "
    "Never introduce any numbers beyond those explicitly provided in the user content. "
    "If a claim cannot be supported numerically from the provided data, omit it. "
    "Prefer concise, analytical sentences (max 25 words). "
    "Any speculative or unsupported statement is a critical error."
)

user_message = f"""
### PERFORMANCE DATA (Facts – do not create new numbers):
- Net Sales: {actual_m['sales']:,.0f}£ (21-Day Trend avg: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Variance: {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%) → Improvement vs trend: {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp ({margin_bps} bps)
- Sales Mix Today (Range/Promo/Clearance): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Sales Mix Trend (Range/Promo/Clearance): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%
- Clearance share change (Trend → Today): {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp
- Sales gap vs trend average: {actual_m['sales'] - (trend_m['sales']/21):,.0f}£

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### TASK (Executive summary with strict constraints):
Write an executive summary as JSON only.

### OUTPUT (JSON, STRICT):
Return ONLY valid JSON with this exact schema:
{{
  "p1": "<string>",
  "p2": "<string>",
  "p3": "<string>",
  "p4": "<string>"
}}
Constraints (mandatory):
- Your entire response MUST be a single JSON object only. Do not output any text before or after the JSON.
- Each of p1, p2, p3, p4 MUST be a single string (not an array). If you cannot fully comply, return empty strings, but NEVER add text outside JSON.
- Each paragraph value MUST be a plain string with EXACTLY 3 sentences.
- Each sentence MUST be 25 words or fewer.
- EVERY sentence MUST include at least one numeric value WITH the correct unit (% / pp / £).
- "p1" MUST start with EXACTLY: "Central Europe's financial performance today".
- Do NOT add any keys or metadata. No markdown, no code fences, no comments, no trailing commas.
- Do NOT copy or restate any line from the Facts section; integrate numbers into sentences only.
- Keep units correct: use "pp" for share changes and "%" for rates; use "£" only for sales amounts.
- Never convert percentages or pp into currency; never convert currency into percentages or pp.
- Do NOT introduce any number that is not present above.
- Do NOT speculate on causes (no macro factors, no marketing execution, no consumer behavior).
- Do NOT use phrases like "could be", "may be", "might", "suggests", or "appears".
- Refer ONLY to "Today" and "21-Day Trend"; no other time horizons.
- Use professional terminology: "margin dilution", "inventory velocity", "baseline demand".

Paragraph-to-key mapping (mandatory logic anchors):
- p1 (CE vs Budget/Forecast and margin improvement):
  • Compare Today Net Sales {actual_m['sales']:,.0f}£ to Trend avg {trend_m['sales']/21:,.0f}£ and state Budget {actual_m['vs_budget_perc']:.2f}% and Forecast {actual_m['vs_forecast_perc']:.2f}% variances.
  • Explain the margin improvement of {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp by referencing the mix shift: Clearance {actual_m['clearance_share']:.1f}% vs {trend_m['clearance_share']:.1f}% (reduced margin dilution).
  • All three sentences must be numeric and within 25 words.

- p2 (HU success vs CZ/SK using ONLY LFL texts and margin rates):
  • Use HU/CZ/SK LFL texts as given and country margin rates (HU {actual_m['countries']['HU']['margin_rate']:.2f}%, CZ {actual_m['countries']['CZ']['margin_rate']:.2f}%, SK {actual_m['countries']['SK']['margin_rate']:.2f}%).
  • Tie stronger HU LFLs to its margin rate numerically; contrast with CZ/SK strictly by numbers.
  • No additional drivers or interpretations.

- p3 (Sales Mix impact on profitability):
  • Quote Today vs Trend mix (Range {actual_m['range_share']:.1f}%/{trend_m['range_share']:.1f}%, Promo {actual_m['promo_share']:.1f}%/{trend_m['promo_share']:.1f}%, Clearance {actual_m['clearance_share']:.1f}%/{trend_m['clearance_share']:.1f}%).
  • State that lower Clearance by {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp reduced margin dilution and supported the {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp margin uplift.
  • Maintain numeric anchoring in all sentences.

- p4 (Departments: risks and opportunities):
  • Use EXACTLY the provided Top/Bottom lists with their % values; restate them compactly with numbers.
  • If a department name explicitly contains 'WIGIG' or 'Seasonal', you may label it as opportunity/risk respectively; otherwise DO NOT assign such labels.
  • Quantify risk/opportunity ONLY by the provided % values; no external justification.
"""

# =========================================
# 5) SEGÉDFÜGGVÉNYEK: health-check, warm-up, JSON kivágás, validáció
# =========================================
def check_ollama_running():
    """Gyors health-check az Ollama szerverre."""
    try:
        r = requests.get(f"{OLLAMA_URL}/api/version", timeout=(3, 3))
        r.raise_for_status()
        return True
    except Exception as e:
        raise RuntimeError(
            f"Ollama server nem érhető el a {OLLAMA_URL} címen.\n"
            f"- Indítsd el: 'ollama serve' (CLI) vagy a Desktop app.\n"
            f"- Hiba: {e}"
        )

def warm_up_model(system_prompt: str):
    """1 tokenes warm-up, hogy a modell betöltődjön (cold start elkerülésére)."""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "Return {}"}
        ],
        "stream": False,
        "options": {
            "format": "json",
            "temperature": 0.15,
            "top_p": 0.9,
            "num_predict": 1,
            "num_ctx": NUM_CTX,
            "seed": SEED,
            "keep_alive": "10m"  # tartsa memóriában
        }
    }
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT))
    r.raise_for_status()

def extract_first_json(text: str) -> str:
    """Kivágja az első teljes JSON objektumot a szövegből (ha a modell köré írt)."""
    start = text.find('{')
    if start == -1:
        raise ValueError("No JSON object start found.")
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError("No complete JSON object found.")

FORBIDDEN_TERMS = [
    "could be", "may be", "might", "suggests", "appears",
    "macroeconomic", "marketing", "consumer behavior"
]
NUMERIC_WITH_UNIT = re.compile(
    r"(\b[-+]?\d[\d.,]*\s*%|\b[-+]?\d[\d.,]*\s*pp|£\s?[-+]?\d[\d.,]*|[-+]?\d[\d.,]*£)"
)
WORD_RE = re.compile(r"\b\w+\b")

def _sentence_word_count(s: str) -> int:
    return len(WORD_RE.findall(s))

def validate_exec_summary_obj(obj: dict):
    """Ellenőrzés: 4×3 mondat; minden mondatban szám+egység; tiltott kifejezések; p1 kezdősor; max 25 szó; £ és pp ne legyen egy kifejezésben."""
    errors = []
    for key in ("p1", "p2", "p3", "p4"):
        if key not in obj:
            errors.append(f"{key} missing")
            continue
        if not isinstance(obj[key], str) or not obj[key].strip():
            errors.append(f"{key} must be a non-empty string")
    if errors:
        return (False, errors)

    for key in ("p1", "p2", "p3", "p4"):
        para = obj[key].strip()

        if key == "p1":
            if not para.startswith("Central Europe's financial performance today"):
                errors.append("p1 must start with: Central Europe's financial performance today")

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", para) if s.strip()]
        if len(sentences) != 3:
            errors.append(f"{key} must have exactly 3 sentences (found {len(sentences)})")

        for i, s in enumerate(sentences, start=1):
            if not NUMERIC_WITH_UNIT.search(s):
                errors.append(f"{key} sentence {i} has no numeric value with unit (% / pp / £)")
            lower = s.lower()
            for bad in FORBIDDEN_TERMS:
                if bad in lower:
                    errors.append(f"{key} sentence {i} contains forbidden term: {bad}")
            if re.search(r"(£\s?[\d.,]+\s*pp|pp\s*£\s?[\d.,]+)", s):
                errors.append(f"{key} sentence {i} mixes £ and pp in the same phrase")
            if _sentence_word_count(s) > 25:
                errors.append(f"{key} sentence {i} exceeds 25 words")

    return (len(errors) == 0, errors)

# =========================================
# 6) P4 FALLBACK + HIBA-ŐR
# =========================================
def build_p4_fallback_from_metrics(actual_metrics: dict) -> str:
    """
    Összerak egy 3 mondatos p4-et a Top/Bottom depts alapján.
    Minden mondat ≤25 szó, van benne szám és helyes egység (% / pp).
    """
    top_items = list(actual_metrics['top_depts'].items())
    bot_items = list(actual_metrics['bottom_depts'].items())

    if len(top_items) < 3 or len(bot_items) < 3:
        t_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in top_items])
        b_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in bot_items])
        s1 = f"Top performers: {t_str}."
        s2 = f"Bottom performers: {b_str}."
        s3 = "The departmental spread highlights inventory velocity and margin dilution considerations with quantified variances."
        return " ".join([s1, s2, s3])

    (t1, t1v), (t2, t2v), (t3, t3v) = top_items[:3]
    (b1, b1v), (b2, b2v), (b3, b3v) = bot_items[:3]
    spread = (t1v - b3v)

    s1 = f"Top performers: {t1} ({t1v:.1f}%), {t2} ({t2v:.1f}%), {t3} ({t3v:.1f}%)."
    s2 = f"Bottom performers: {b1} ({b1v:.1f}%), {b2} ({b2v:.1f}%), {b3} ({b3v:.1f}%)."
    s3 = f"The spread between top {t1v:.1f}% and bottom {b3v:.1f}% is {spread:.1f} pp, indicating inventory velocity and margin dilution risks."

    def trim_to_25_words(tx: str) -> str:
        parts = tx.split()
        return " ".join(parts[:25])

    p4 = " ".join([trim_to_25_words(s1), trim_to_25_words(s2), trim_to_25_words(s3)])
    return p4

def _only_p4_errors(errs: list) -> bool:
    """True, ha minden hiba p4-re vonatkozik → használható a p4-fallback."""
    if not errs:
        return False
    for e in errs:
        el = e.lower().strip()
        if not (el.startswith("p4 ") or el == "p4 missing" or "p4" in el):
            return False
    return True

# =========================================
# 7) MODELLHÍVÁSOK (JSON) + JAVÍTÓKÖR
# =========================================
def _call_model_json(system_prompt: str, user_message: str) -> str:
    """Ollama /api/chat hívás JSON módra optimalizált opciókkal és keep_alive-val."""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message + "\n\nReturn ONLY JSON as specified."}
        ],
        "stream": False,
        "options": {
            "format": "json",
            "temperature": 0.15,
            "top_p": 0.9,
            "num_predict": NUM_PREDICT,
            "num_ctx": NUM_CTX,
            "seed": SEED,
            "repeat_penalty": 1.18,
            "repeat_last_n": 256,
            "presence_penalty": 0.0,
            "frequency_penalty": 0.1,
            "keep_alive": "10m"
        }
    }
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT))
    r.raise_for_status()
    data = r.json()
    return data.get("message", {}).get("content", "")

def analyse_financial_summaries_json(system_prompt: str, user_message: str):
    """
    1) Szerver ellenőrzés + warm-up.
    2) JSON bekérés (p1..p4).
    3) Kivágás, parse, validáció.
    4) Egyszeri javítókör hiba esetén.
    5) Ha továbbra is csak p4 hibás → p4 programozott fallback.
    6) 4 bekezdés összefűzése és visszaadása.
    """
    check_ollama_running()
    try:
        warm_up_model(system_message)
    except requests.exceptions.ReadTimeout:
        pass  # warm-up időtúllépés esetén is megyünk tovább

    # Első kör
    raw = _call_model_json(system_prompt, user_message)

    # JSON kivágás + parse
    try:
        json_str = extract_first_json(raw)
    except ValueError:
        json_str = raw  # hátha maga a teljes válasz már JSON

    try:
        obj = json.loads(json_str)
    except json.JSONDecodeError:
        raise ValueError("Model did not return valid JSON in first attempt.")

    ok, errs = validate_exec_summary_obj(obj)

    # Ha csak p4 problémás → p4 fallback és kész
    if not ok and _only_p4_errors(errs):
        obj["p4"] = build_p4_fallback_from_metrics(actual_m)
        final_text = "\n\n".join([obj.get("p1",""), obj.get("p2",""), obj.get("p3",""), obj.get("p4","")])
        return final_text

    # Javítókör (egyszeri)
    if not ok:
        repair_instructions = (
            "Your previous JSON failed these checks: "
            + "; ".join(errs)
            + ". Fix and return ONLY valid JSON with keys p1, p2, p3, p4 (each a single string). "
              "No markdown, no arrays, no extra keys, no comments."
        )

        payload = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message + "\n\nReturn ONLY JSON as specified."},
                {"role": "assistant", "content": json_str},
                {"role": "user", "content": repair_instructions}
            ],
            "stream": False,
            "options": {
                "format": "json",
                "temperature": 0.15,
                "top_p": 0.9,
                "num_predict": NUM_PREDICT,
                "num_ctx": NUM_CTX,
                "seed": SEED,
                "repeat_penalty": 1.18,
                "repeat_last_n": 256,
                "presence_penalty": 0.0,
                "frequency_penalty": 0.1,
                "keep_alive": "10m"
            }
        }
        r2 = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT))
        r2.raise_for_status()
        content2 = r2.json().get("message", {}).get("content", "")

        try:
            json_str2 = extract_first_json(content2)
        except ValueError:
            json_str2 = content2

        obj2 = json.loads(json_str2)
        ok2, errs2 = validate_exec_summary_obj(obj2)

        # Ha a második körben is csak p4 a gond → p4 fallback és kész
        if not ok2 and _only_p4_errors(errs2):
            obj2["p4"] = build_p4_fallback_from_metrics(actual_m)
            final_text = "\n\n".join([obj2.get("p1",""), obj2.get("p2",""), obj2.get("p3",""), obj2.get("p4","")])
            return final_text

        if not ok2:
            raise ValueError("Model failed after repair: " + "; ".join(errs2))

        obj = obj2

    # Siker esetén 4 bekezdés összefűzése
    final_text = "\n\n".join([obj["p1"], obj["p2"], obj["p3"], obj["p4"]])
    return final_text

# =========================================
# 8) FUTTATÁS + KIÍRÁS
# =========================================
total_summary = analyse_financial_summaries_json(
    system_prompt=system_message,
    user_message=user_message
)

def write_out(content):
    output_file = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_final_v21.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content.strip())
    print(f"Report saved: {output_file}")

write_out(total_summary)


C:\Users\pmajor1\AppData\Local\Temp\ipykernel_40248\2670221581.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(
C:\Users\pmajor1\AppData\Local\Temp\ipykernel_40248\2670221581.py:76: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(


Report saved: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_final_v21.md


In [3]:
total_summary

' {\n    "p1": "Central Europe\'s financial performance today is mixed, but there are signs of improvement.",\n    "p2": "Hungary performed strongly compared to Czechia and Slovakia, driven primarily by higher margins.",\n    "p3": "The reduction in clearance sales contributed positively towards this outcome.",\n    "p4": [\n        "Top performers include WIGIG at +58.5%, Electricals at +19.2%, and Shopping bag at +16.9%.",\n        "Home seasonal was weakest at -23.5%, followed closely by Gardening at -38.9% and Other GM at -43.3%"\n    ]\n}'

### Ez nem működött rosszul:
user_message = f"""
### PERFORMANCE DATA (Facts – do not create new numbers):
- Net Sales: {actual_m['sales']:,.0f}£ (21-Day Trend avg: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Variance: {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%) → Improvement vs trend: {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp ({margin_bps} bps)
- Sales Mix Today (Range/Promo/Clearance): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Sales Mix Trend (Range/Promo/Clearance): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%
- Clearance share change (Trend → Today): {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp
- Sales gap vs trend average: {actual_m['sales'] - (trend_m['sales']/21):,.0f}£

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### TASK (Executive summary with strict constraints):
Write EXACTLY 4 paragraphs, EACH with EXACTLY 3 sentences.
EVERY sentence MUST reference at least ONE numeric fact from the sections above WITH its correct unit (% / pp / £).
Each sentence MUST be 25 words or fewer.
Integrate the numbers directly into the sentences (no lists, no bullets).
Use professional terminology: "margin dilution", "inventory velocity", "baseline demand".
Do NOT introduce any number that is not present above.
Do NOT speculate on causes (no macro factors, no marketing execution, no consumer behavior).
Do NOT use phrases like "could be", "may be", "might", "suggests", or "appears".
Refer ONLY to "Today" and "21-Day Trend"; no other time horizons.
Do NOT copy or restate any line from the Facts section; integrate numbers into sentences only.
Always keep units correct: use "pp" for share changes and "%" for rates; use "£" only for sales amounts.
Never convert percentages or pp values into currency; never convert currency into percentages or pp.
Do NOT output headings, lists, bullets, or labels like "Sales Mix Today"; write paragraphs only.

### PARAGRAPH-BY-PARAGRAPH LOGIC ANCHORS (Mandatory):
1) CE vs Budget/Forecast and margin improvement:
   - Compare Today Net Sales {actual_m['sales']:,.0f}£ to Trend avg {trend_m['sales']/21:,.0f}£ and state Budget {actual_m['vs_budget_perc']:.2f}% and Forecast {actual_m['vs_forecast_perc']:.2f}% variances.
   - Explain the margin improvement of {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp by referencing the mix shift: Clearance {actual_m['clearance_share']:.1f}% vs {trend_m['clearance_share']:.1f}% (reduced margin dilution).
   - Keep all three sentences numeric and within 25 words.

2) HU success vs CZ/SK using ONLY LFL texts and margin rates:
   - Use HU/CZ/SK Range/Promo/Clearance LFL texts as given, and country margin rates (HU {actual_m['countries']['HU']['margin_rate']:.2f}%, CZ {actual_m['countries']['CZ']['margin_rate']:.2f}%, SK {actual_m['countries']['SK']['margin_rate']:.2f}%).
   - Tie stronger HU LFLs to its higher/lower margin rate numerically; contrast with CZ/SK figures strictly by numbers.
   - No additional drivers or interpretations; stay on LFL and margin rate numbers.

3) Sales Mix impact on profitability:
   - Quote Today vs Trend mix (Range {actual_m['range_share']:.1f}%/{trend_m['range_share']:.1f}%, Promo {actual_m['promo_share']:.1f}%/{trend_m['promo_share']:.1f}%, Clearance {actual_m['clearance_share']:.1f}%/{trend_m['clearance_share']:.1f}%).
   - State that lower Clearance by {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp reduced margin dilution and supported the {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp margin uplift.
   - Maintain numeric anchoring in all sentences.

4) Departments: risks and opportunities:
   - Use EXACTLY the provided Top/Bottom lists with their % values; restate them compactly with numbers.
   - If a department name explicitly contains 'WIGIG' or 'Seasonal', you may label it as opportunity/risk respectively; otherwise DO NOT assign such labels.
   - Quantify risk/opportunity ONLY by the provided % values; no external justification.


### OUTPUT FORMAT (MANDATORY):
- Return only the 4 paragraphs (no headings, no bullets, no extra lines).
- Each paragraph separated by a single blank line.
- Start with EXACTLY: Central Europe's financial performance today
- End with the literal token: <END_OF_REPORT>

"""

In [6]:
actual_m['countries']['HU']['range_lfl_txt']

'11.8% increase'

## Prompts what actually not working well, but I want to keep them

In [ ]:
top_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['top_depts'].items()])
bottom_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['bottom_depts'].items()])

# Itt visszatesszük a teljes kontextust
prompt = f"""### SYSTEM ROLE: Senior Financial Analyst
### PERFORMANCE DATA (Today vs 21-Day Trend):
- Net Sales: {actual_m['sales']:,.0f}£ (Trend: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Variance: {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%)
- Sales Mix (Range/Promo/Cl.): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Trend Mix (Range/Promo/Cl.): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%

### REGIONAL LFLs:
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### DEPARTMENTS:
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### TASK:
Write a high-level executive summary (300-400 words). 
1. Paragraph 1: Analyze CE level performance vs Budget and Forecast. Explain the **{actual_m['margin_rate'] - trend_m['margin_rate']:.2f} percentage point** margin improvement.
2. Paragraph 2: Deep-dive into HU's success vs CZ/SK struggles. Connect this to the LFL figures.
3. Paragraph 3: Explain the impact of the Sales Mix shift. Specifically, how the reduction in Clearance share ({actual_m['clearance_share']:.1f}% vs {trend_m['clearance_share']:.1f}%) protected the bottom line.
4. Paragraph 4: Highlight department risks and opportunities (WIGIG vs Seasonal).

### RULES:
- NO pleasantries. NO "Here is your report". 
- INTEGRATE the numbers into the sentences.
- Use professional terminology: "margin dilution", "inventory velocity", "baseline demand".

### REPORT START:
Central Europe's financial performance today"""

In [8]:
user_message=f"""
Compare TODAY vs the 21-DAY TREND. 

### SECTION 1: CE OVERVIEW
- Net Sales: Today {actual_m['sales']:,.0f}£ (Trend Avg: {trend_m['sales']/21:,.0f}£)
- Budget Var: Today {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Var: Today {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: Today {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%)
- Sales Mix (Range/Promo/Clearance Share): Today {actual_m['range_share']:.1f}%/{actual_m['promo_share']:.1f}%/{actual_m['clearance_share']:.1f}% vs Trend {trend_m['range_share']:.1f}%/{trend_m['promo_share']:.1f}%/{trend_m['clearance_share']:.1f}%

### SECTION 2: COUNTRY PERFORMANCE (HU, CZ, SK)
- HU: Range LFL: {actual_m['countries']['HU']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range LFL: {actual_m['countries']['CZ']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range LFL: {actual_m['countries']['SK']['range_lfl_txt']}, Promo LFL: {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance LFL: {actual_m['countries']['SK']['clearance_lfl_txt']}

### INSTRUCTIONS:
1. Identify the main CE driver (HU, CZ or SK).
2. Pay close attention to the "increase/decrease" tags provided in the data. Never interpret an "increase" as a negative performance.
3. Analyze the 'Sales Mix shift': Is the margin rate improving or eroding due to the Promo/Clearance mix?
4. For each country, provide a 1-sentence insight explaining if their performance is healthy (range-driven) or risky (clearance-driven).

OUTPUT FORMAT:
- Use clear headings: # CE Overview, # Key Regional Drivers, # Country Insights.
- Professional financial narrative. Bold the most important percentages.
"""

### Version 4

In [6]:
from core import utils, database, paths
import pandas as pd
import datetime
import os

# 1. ADATELŐKÉSZÍTÉS
data = pd.read_csv(paths.ASSETS_PATH + r"\daily_test_data.csv")
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]

def get_complete_metrics(df: pd.DataFrame):
    """Kiszámolja az összesített, országos és osztály szintű mutatókat."""
    
    def calc_lfl_info(ty_col, ly_col, input_df=None):
        target_df = input_df if input_df is not None else df
        ty_sum = target_df[ty_col].sum()
        ly_sum = target_df[ly_col].sum()
        if ly_sum == 0: return "0.0% flat"
        val = (ty_sum / ly_sum - 1) * 100
        direction = "increase" if val >= 0 else "decrease"
        return f"{val:.1f}% {direction}"

    # --- TOTAL CE SZINT ---
    total_sales = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
    total_margin = df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum()
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()
    
    metrics = {
        'sales': total_sales,
        'margin_rate': (total_margin / total_sales * 100) if total_sales != 0 else 0,
        'vs_budget_perc': ((total_sales / budget - 1) * 100) if budget != 0 else 0,
        'vs_forecast_perc' : ((total_sales / forecast - 1) * 100) if forecast != 0 else 0,
        'range_share' : (df['sales_ex_vat_ty_standard'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'promo_share': (df['sales_ex_vat_ty_promo'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'clearance_share': (df['sales_ex_vat_ty_clearance'].sum() / total_sales * 100) if total_sales != 0 else 0,
    }

    # --- ORSZÁGOS SZINT ---
    metrics['countries'] = {}
    for c in ['HU', 'CZ', 'SK']:
        c_df = df[df['country'] == c]
        if not c_df.empty:
            c_sales = c_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['countries'][c] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard', c_df),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo', c_df),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance', c_df),
                'margin_rate': (c_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if c_sales != 0 else 0
            }

    # --- DEPARTMENT SZINT (Top/Bottom 3) ---
    dept_gb = df.groupby('department_name').apply(
        lambda x: (x['sales_lfl_standard'].sum() / x['sales_lflb_standard'].sum() - 1) * 100 if x['sales_lflb_standard'].sum() != 0 else 0
    ).sort_values(ascending=False)
    
    metrics['top_depts'] = dept_gb.head(3).to_dict()
    metrics['bottom_depts'] = dept_gb.tail(3).to_dict()

    return metrics

actual_m = get_complete_metrics(actual_df)
trend_m = get_complete_metrics(trend_df)


taken_outs = "Avoid narrative, storytelling, and speculation. "

system_message = ("""
You are a Senior Financial Controller. Your ONLY task is to populate a performance report using the provided user data. 

STRICT OUTPUT FORMAT:
Return a JSON object with 4 keys: "Highlights", "Sales_vs_Forecast", "Sales_Drivers", "Margin_Analysis". 
The values MUST be single, long strings (sentences), NOT lists.

NARRATIVE GUIDELINES:
- In "Highlights", replace all placeholders with actual Sales, LFL%, and Country metrics.
- In "Sales_vs_Forecast", calculate the variance and name the specific driving departments provided in the data.
- Do NOT use "..." or placeholders. If a specific data point is missing, omit that part of the sentence.
- Follow the narrative style: "Metric was [Value], higher/lower than [Benchmark] by [Variance]."

JSON STRUCTURE TO FILL:
{
  "Highlights": "Complete sentence here...",
  "Sales_vs_Forecast": "Complete sentence here...",
  "Sales_Drivers": "Complete sentence here...",
  "Margin_Analysis": "Complete sentence here...",
  "On department level: "Complete sentence here..."
}
""")


# Csak azokat az adatokat adjuk át, amik kellenek a szövegbe, 
# és megmutatjuk a modellnek a várt stílust.
top_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['top_depts'].items()])
bottom_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['bottom_depts'].items()])


# Számítsunk ki 2-3 kulcsfontosságú összefüggést Pythonban, hogy ne az LLM-nek kelljen
margin_diff = actual_m['margin_rate'] - trend_m['margin_rate']
mix_improvement = trend_m['clearance_share'] - actual_m['clearance_share']

margin_bps = int((actual_m['margin_rate'] - trend_m['margin_rate']) * 100)

# KIZÁRÓLAG a tények, nulla magyarázat a modellnek, hogy mit csináljon a tagekkel
margin_improvement = actual_m['margin_rate'] - trend_m['margin_rate']
clearance_drop = trend_m['clearance_share'] - actual_m['clearance_share']
sales_gap = actual_m['sales'] - (trend_m['sales']/21)


# Itt visszatesszük a teljes kontextust



user_message = f"""
### PERFORMANCE DATA (Facts – do not create new numbers):
- Net Sales: {actual_m['sales']:,.0f}£ (21-Day Trend avg: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget_perc']:.2f}% (Trend: {trend_m['vs_budget_perc']:.2f}%)
- Forecast Variance: {actual_m['vs_forecast_perc']:.2f}% (Trend: {trend_m['vs_forecast_perc']:.2f}%)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%) → Improvement vs trend: {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp ({margin_bps} bps)
- Sales Mix Today (Range/Promo/Clearance): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Sales Mix Trend (Range/Promo/Clearance): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%
- Clearance share change (Trend → Today): {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp
- Sales gap vs trend average: {actual_m['sales'] - (trend_m['sales']/21):,.0f}£

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### OUTPUT:
Constraints (mandatory):
- Each paragraph value MUST be a plain string with EXACTLY 3 sentences.
- Each sentence MUST be 25 words or fewer.
- EVERY sentence MUST include at least one numeric value WITH the correct unit (% / pp / £).
- Do NOT add any keys or metadata. No markdown, no code fences, no comments, no trailing commas.
- Do NOT copy or restate any line from the Facts section; integrate numbers into sentences only.
- Keep units correct: use "pp" for share changes and "%" for rates; use "£" only for sales amounts.
- Never convert percentages or pp into currency; never convert currency into percentages or pp.
- Do NOT introduce any number that is not present above.
- Do NOT speculate on causes (no macro factors, no marketing execution, no consumer behavior).
- Do NOT use phrases like "could be", "may be", "might", "suggests", or "appears".
- Refer ONLY to "Today" and "21-Day Trend"; no other time horizons.
- Use professional terminology: "margin dilution", "inventory velocity", "baseline demand".

Paragraph-to-key mapping (mandatory logic anchors):
- p1 (CE vs Budget/Forecast and margin improvement):
  • Compare Today Net Sales {actual_m['sales']:,.0f}£ to Trend avg {trend_m['sales']/21:,.0f}£ and state Budget {actual_m['vs_budget_perc']:.2f}% and Forecast {actual_m['vs_forecast_perc']:.2f}% variances.
  • Explain the margin improvement of {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp by referencing the mix shift: Clearance {actual_m['clearance_share']:.1f}% vs {trend_m['clearance_share']:.1f}% (reduced margin dilution).
  • All three sentences must be numeric and within 25 words.

- p2 (HU success vs CZ/SK using ONLY LFL texts and margin rates):
  • Use HU/CZ/SK LFL texts as given and country margin rates (HU {actual_m['countries']['HU']['margin_rate']:.2f}%, CZ {actual_m['countries']['CZ']['margin_rate']:.2f}%, SK {actual_m['countries']['SK']['margin_rate']:.2f}%).
  • Tie stronger HU LFLs to its margin rate numerically; contrast with CZ/SK strictly by numbers.
  • No additional drivers or interpretations.

- p3 (Sales Mix impact on profitability):
  • Quote Today vs Trend mix (Range {actual_m['range_share']:.1f}%/{trend_m['range_share']:.1f}%, Promo {actual_m['promo_share']:.1f}%/{trend_m['promo_share']:.1f}%, Clearance {actual_m['clearance_share']:.1f}%/{trend_m['clearance_share']:.1f}%).
  • State that lower Clearance by {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp reduced margin dilution and supported the {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp margin uplift.
  • Maintain numeric anchoring in all sentences.

- p4 (Departments: risks and opportunities):
  • Use EXACTLY the provided Top/Bottom lists with their % values; restate them compactly with numbers.
  • Quantify risk/opportunity ONLY by the provided % values; no external justification.
"""





# A függvényed hívása (ügyelj rá, hogy a payload-ban a 'messages' listát használd)
total_summary = analyse_financial_summaries(
    system_prompt=system_message, 
    user_data=user_message,
    model='martain7r/finance-llama-8b:q4_k_m'
)


def write_out(content):
    output_file = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_final_VERSION4_v7.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről
    print(f"Report saved: {output_file}")

write_out(total_summary)

C:\Users\pmajor1\AppData\Local\Temp\ipykernel_35276\177926842.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(
C:\Users\pmajor1\AppData\Local\Temp\ipykernel_35276\177926842.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(


Report saved: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_final_VERSION4_v7.md
